In [39]:
from itertools import combinations
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, logging, RobertaModel
import torch
import torch.nn.functional as F

from dataset import CL_Dataset, CustomSubset
from coval.eval import evaluator
from model import SiameseNetwork, CustomContrastiveLoss
from report_metrics import compute_metrics_cdcr

logging.set_verbosity_error()

In [2]:
## Parameters
#Data

train_conll_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
val_conll_paths = ['../CDCR/data/ECBplus-prep/output_data/val']
mention_type = "event"
context_method = "hsu"
train_dataset_name = "ecbplus_train"
val_dataset_name = "ecbplus_val"
language_model = 'roberta-base'
special_tokens = ('[E]', '[/E]')

#Model
gpu_num = [0]
margin=0.4

#Optimizer and learning
learning_rate = 2e-7
weight_decay = 1e-2
num_epochs = 100
train_batch_size = 4
val_batch_size = 32

In [3]:
#Initialize model and tokenizer
roberta = RobertaModel.from_pretrained(pretrained_model_name_or_path= language_model) 
tokenizer = RobertaTokenizer.from_pretrained(language_model)
tokenizer.add_tokens(special_tokens)

2

In [4]:
#Initialization
#Load data

cl_cdcr_train = CL_Dataset(name=train_dataset_name,data_paths=train_conll_paths,context_method=context_method,
                           mention_type=mention_type,tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="train") 

cl_cdcr_val = CL_Dataset(name=val_dataset_name,data_paths=val_conll_paths,context_method=context_method,
                           mention_type=mention_type,tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="inference") 

#Use subset for testing:
#cl_cdcr_val = CustomSubset(cl_cdcr_val,[0,1,2,3])

#Keep in mind ECB topic repeated in train and topic. topic 34 I think

train_dataloader= DataLoader(cl_cdcr_train, batch_size=train_batch_size, shuffle=True)

val_dataloader= DataLoader(cl_cdcr_val, batch_size=val_batch_size, shuffle=False)

In [5]:
# Initialize model
device_ids = gpu_num
device = torch.device("cuda:{}".format(device_ids[0]))
#Temporary use of cpu
device = torch.device("cpu")

siamese_model_single = SiameseNetwork(model=roberta,tokenizer=tokenizer).to(device)
model=siamese_model_single
# To paralelize
#siamese_model = torch.nn.DataParallel(siamese_model_single, device_ids=device_ids)
#model = siamese_model_single


#

In [6]:
# Initialize optimizer
#Parameters
# Define the learning rate and weight decay

# Define optimizer!
# AdamW optimizer
# Epochs: 100 Events 50 Entities
# Batch size 16
# m 0.4 0.7
# Clistering Threshold
# Learning rate  2e-7 2e-7


optimizer_params = [
    {'params': model.language_model.parameters(), 'lr': learning_rate},
    {'params': model.mlp_cls_embed.parameters(), 'lr': learning_rate},
    {'params': model.mlp_mention_embed.parameters(), 'lr': learning_rate},
]

contrastive_loss = CustomContrastiveLoss(margin=margin)
optimizer = torch.optim.AdamW(optimizer_params)

In [7]:
#Store vector representations by topic
dict_embeddings_by_topic={}
#Validation dataloader
for indices, array_mention in val_dataloader: #Validation batch size should be different. It will be way faster
    # I also need to get the keys from the mentions to cluster them later
    with torch.no_grad():
        output_mentions=model(array_mention)
        batch_keys = [cl_cdcr_val.keys_list[i] for i in indices]

        for key, embedding in zip(batch_keys,output_mentions):
            mention_temp = cl_cdcr_val.cdcr_dataset.mentions[key]
            topic_number = mention_temp.topic_id

            #Stores embeddings by topic
            if topic_number in dict_embeddings_by_topic:
                dict_embeddings_by_topic[topic_number][key] = embedding
            else:
                dict_embeddings_by_topic[topic_number] = {key: embedding}

In [8]:
#Load predicted clusters
predicted_clusters_df_list = []
max_predicted_cluster_value = 0

for topic, dict_topic in dict_embeddings_by_topic.items():
    list_keys_dict_topic = list(dict_topic.keys())
    list_of_embeddings = [dict_topic[key] for key in list_keys_dict_topic]
    #Computes pairwise cosine similarity
    distances = torch.tensor([[1 - F.cosine_similarity(vec1, vec2, dim=0).item() for vec2 in list_of_embeddings] for vec1 in list_of_embeddings])

    #Clustering
    clustering = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average',
                                            distance_threshold=0.2)
    fit = clustering.fit(distances)

    #Implement CoNLL score with this fit. I gotta merge on mention_id
    #Save as Dataframe
    adjusted_predicted_label = fit.labels_+ max_predicted_cluster_value
    predicted_df_by_topic = pd.DataFrame({"mention_id":list_keys_dict_topic,"predicted_cluster":adjusted_predicted_label,"topic_id":[topic]*len(list_keys_dict_topic)})
    predicted_clusters_df_list.append(predicted_df_by_topic)

    max_predicted_cluster_value = max(adjusted_predicted_label) +1

predicted_clusters_df = pd.concat(predicted_clusters_df_list)

In [9]:
#Load gold clusters
# Your loop to create the list of dictionaries
mentions_data = []  

for k, v in cl_cdcr_val.cdcr_dataset.mentions.items():
    mention_id = k
    topic_id = v.topic_id
    subtopic_id = v.subtopic_id  # Might be missing in NewsWCL50
    coref_chain = v.coref_chain
    mention_type = v.type

    # Create a dictionary to store the values for each mention
    mention_dict = {
        "mention_id": mention_id,
        "subtopic_id": subtopic_id,
        "coref_chain": coref_chain,
        "mention_type": mention_type
    }

    # Append the dictionary to the list
    mentions_data.append(mention_dict)

# Create a DataFrame from the list of dictionaries
gold_clusters_df = pd.DataFrame(mentions_data)

set_coref_chains = gold_clusters_df.coref_chain.unique()
enum_dict_coref = {item: index for index, item in enumerate(set_coref_chains)}
gold_clusters_df["gold_cluster"] = gold_clusters_df["coref_chain"].map(enum_dict_coref)

In [10]:
merged_df = gold_clusters_df.merge(predicted_clusters_df,how="inner", on="mention_id")

In [11]:
merged_df

,mention_id,subtopic_id,coref_chain,mention_type,gold_cluster,predicted_cluster,topic_id
0,12_10ecb_ACT17403639225065902_0_4_F4LF,12ecb,ACT17403639225065902,event,575,0,12
1,12_11ecb_ACT17403639225065902_0_5_TERD,12ecb,ACT17403639225065902,event,575,0,12
2,12_12ecb_ACT17403639225065902_0_23_VvP8,12ecb,ACT17403639225065902,event,575,0,12
3,12_13ecb_ACT17403639225065902_0_3_i5A7,12ecb,ACT17403639225065902,event,575,0,12
4,12_14ecb_ACT17403639225065902_0_4_DtDL,12ecb,ACT17403639225065902,event,575,0,12
...,...,...,...,...,...,...,...
981,5_9ecbplus_ACT16702401360576669_10_22_UoAo,5ecbplus,ACT16702401360576669,event,725,7,5
982,5_4ecbplus_ACT16704969063922956_3_8_VH9e,5ecbplus,ACT16704969063922956,event,726,7,5
983,5_5ecbplus_ACT16702238123414525_7_3_dpjx,5ecbplus,ACT16702238123414525,event,727,7,5
984,5_6ecbplus_ACT28256774522991681_11_20_ipHz,5ecbplus,ACT28256774522991681,event,728,7,5


In [19]:
cl_cdcr_val.cdcr_dataset.name

'ecbplus_val'

In [29]:
import os

def format_line(dataset_name,row, cluster):
    return f"{dataset_name}\t({row[cluster]})"


def write_output_to_file(df, dataset_name, output_file_name, cluster_col_name):
    # Construct the output directory path
    output_directory_path = os.path.join(os.getcwd(), "conll_files")

    # Ensure the output directory exists; if not, create it
    if not os.path.exists(output_directory_path):
        os.makedirs(output_directory_path)

    output_file = os.path.join(output_directory_path, output_file_name)
    with open(output_file, "w") as f:
        f.write(f"#begin document ({dataset_name}); part 000\n")
        for _, row in df.iterrows():
            f.write(format_line(dataset_name, row, cluster_col_name) + "\n")
        f.write("#end document\n")

    print(f"File '{output_file}' written successfully.")

In [43]:
key_filename=f"key_{cl_cdcr_val.cdcr_dataset.name}.conll"

In [31]:
write_output_to_file(merged_df,cl_cdcr_val.cdcr_dataset.name,key_filename,"gold_cluster")

File 'c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\CL_COREF\conll_files\key_ecbplus_val.conll' written successfully.


In [35]:
sys_filename=f"key_{cl_cdcr_val.cdcr_dataset.name}.conll"

In [36]:
write_output_to_file(merged_df,cl_cdcr_val.cdcr_dataset.name,sys_filename,"predicted_cluster")

File 'c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\CL_COREF\conll_files\key_ecbplus_val.conll' written successfully.


In [49]:
allmetrics = [('mentions', evaluator.mentions), ('muc', evaluator.muc),
        ('bcub', evaluator.b_cubed), ('ceafe', evaluator.ceafe),
        ('lea', evaluator.lea)]

metrics = allmetrics

NP_only = False
remove_nested = False
keep_singletons=True
min_span=False

In [57]:
#Report metrics
computed_metrics = compute_metrics_cdcr(os.path.join(os.getcwd(), "conll_files",key_filename),
                     os.path.join(os.getcwd(), "conll_files",sys_filename),
                       metrics, NP_only, remove_nested,
        keep_singletons, min_span)

computed_metrics = computed_metrics.set_index("Metric")

In [59]:
conll=computed_metrics.loc["muc", "F1"] + computed_metrics.loc["bcub", "F1"] + computed_metrics.loc["ceafe", "F1"]
conll = (conll / 3)
#Is this the right formula?


In [ ]:
conll

In [25]:
# Turn coreference chains into strings to feed the conll file only with integers
#I can do it after the merge
#initializes max cluster value in 0
# increases each cluster the max cluster value
# updates max cluster to the maximum of each cluster

'12_10ecb_HUM17403498594960941_0_7_dZMa'

In [ ]:
#Training loop
for epoch in range(num_epochs):
    accumulate_loss = 0
    model.train()

    #Train dataloader
    for array_mention1,array_mention2, labels in train_dataloader:


        optimizer.zero_grad()  # Zero the gradients
        output_mentions1 = model(array_mention1)
        output_mentions2 = model(array_mention2)
        
        loss = contrastive_loss(output_mentions1, output_mentions2, labels)

        loss.backward()  # Backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()  # Optimization step

        torch.cuda.empty_cache()
        accumulate_loss += loss.item()

    #Validation
    model.eval()
    #Store vector representations by topic
    dict_embeddings_by_topic={}
    #Validation dataloader
    for indices, array_mention in val_dataloader: #Validation batch size should be different. It will be way faster
        # I also need to get the keys from the mentions to cluster them later
        with torch.no_grad():
            output_mentions=model(array_mention)

            batch_keys = [cl_cdcr_val.keys_list[i] for i in indices]

            for key, embedding in zip(batch_keys,output_mentions):
                mention_temp = cl_cdcr_val.cdcr_dataset.mentions[key]
                topic_number = mention_temp.topic_id

                #Stores embeddings by topic
                if topic_number in dict_embeddings_by_topic:
                    dict_embeddings_by_topic[topic_number][key] = embedding
                else:
                    dict_embeddings_by_topic[topic_number] = {key: embedding}
        
        #Once I store all the values by topic, I run the agglomerative clustering by topic and compute F1-CoNLL score
            



    # Optionally, you can print or log the loss for monitoring training progress
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')